# Delta Chip Extraction
## GEOG 6160 Final Project
## Magnus Tveit

**Run `PrepData.ipynb` first** to ensure `Data/` is populated.

Reads polygons from the shapefile and extracts 8×8 pixel chips from each
training year's 7-band Landsat stack. Chips are saved as `.npy` files in the folder
structure expected by `DeltaCNN.ipynb`.

The shapefile was created in ArcGIS pro

Shapefile field schema:
```
type : 0 = not_delta,  1 = delta
year : 00 = applies to all training years
       13 / 16 / 20 / 22 = year-specific polygon
```

Output structure:
```
Chips/
   train/
      delta/
      not_delta/
   valid/
      delta/
      not_delta/
   test/
      delta/ 
      not_delta/
```

> **Note:** Delete the `Chips/` folder entirely before re-running if you change anything,
> otherwise old and new chips will be mixed.

### Set Up

In [1]:
import os, random
import numpy as np
import rasterio
from rasterio.features import geometry_mask
import geopandas as gpd

### Settings
> Note: Update the `attempt` to match the `TrainingPolygons/` subfolder

In [2]:
# Attempt number; relates to the folder in which the training polygons Shapefiles are stored
attempt = 2

random.seed(42)
np.random.seed(42)

# Paths — run this notebook from inside Delta/
base_dir  = os.getcwd()
data_dir  = os.path.join(base_dir, "..", "Data")   # shared Data/ one level up
shp_path  = os.path.join(base_dir, "TrainingPolygons", str(attempt), "TrainingPolygons.shp")
chips_dir = os.path.join(base_dir, "Chips")

CHIP_SIZE                = 8     # pixel width/height of each chip
MAX_NOT_DELTA_MULTIPLIER = 4     # cap not_delta chips at this multiple of delta count
TRAINING_YEARS           = [13, 16, 20, 22]
COVERAGE_THRESHOLD       = 0.5   # chip must be >50% inside the polygon to be kept
CLASSES                  = {0: "not_delta", 1: "delta"}
BAND_SUFFIXES            = ["_SR_B2.TIF", "_SR_B3.TIF", "_SR_B4.TIF",
                             "_SR_B5.TIF", "_SR_B6.TIF", "_SR_NDVI.TIF", "_SR_NDWI.TIF"]
SCALE, OFFSET            = 0.0000275, -0.2   # Landsat Collection 2 surface reflectance scaling
TRAIN_RATIO, VALID_RATIO = 0.70, 0.15        # remaining 15% goes to test

# Create output folder structure (safe to re-run — won't delete existing chips)
for split in ["train", "valid", "test"]:
    for cls in CLASSES.values():
        os.makedirs(os.path.join(chips_dir, split, cls), exist_ok=True)

# Load shapefile and reproject to match Landsat CRS
os.environ["SHAPE_RESTORE_SHX"] = "YES"   # auto-rebuild missing .shx index
gdf = gpd.read_file(shp_path).to_crs("EPSG:32612")
print(f"  {len(gdf)} polygons loaded")
print(f"  year values : {sorted(gdf['year'].unique())}")
print(f"  type values : {sorted(gdf['type'].unique())}\n")

  36 polygons loaded
  year values : [np.int32(0), np.int32(13), np.int32(16), np.int32(20), np.int32(22)]
  type values : ['0', '1']



### Extract Chips
Slides an 8×8 grid across each polygon and keeps chips with >50% coverage. Delta chips are augmented ×6 to try to help balance the dataset.

In [3]:
def extract_chips(stacked, transform, polygon, chip_size, coverage_threshold):
    """Yield (chip_size, chip_size, 7) chips whose centre falls inside the polygon."""
    n_bands, H, W = stacked.shape
    half = chip_size // 2

    # Rasterize the polygon to a boolean mask matching the image grid
    inside = geometry_mask([polygon.__geo_interface__],
                           out_shape=(H, W), transform=transform, invert=True)
    rows, cols = np.where(inside)
    if len(rows) == 0:
        return

    seen = set()
    for r, c in zip(rows, cols):
        # Snap to the nearest grid-aligned chip origin
        r_s = (r // half) * half
        c_s = (c // half) * half
        if (r_s, c_s) in seen:
            continue
        seen.add((r_s, c_s))
        r0, r1 = r_s - half, r_s + half
        c0, c1 = c_s - half, c_s + half

        # Skip chips that extend outside the image boundary
        if r0 < 0 or c0 < 0 or r1 > H or c1 > W:
            continue

        # Skip chips where fewer than 50% of pixels are inside the polygon
        if inside[r0:r1, c0:c1].mean() < coverage_threshold:
            continue

        chip = stacked[:, r0:r1, c0:c1]
        yield np.moveaxis(chip, 0, -1)   # reshape to (chip_size, chip_size, bands)


def augment_chip(chip):
    """Return 6 versions of a chip: original + 3 rotations + 2 flips."""
    aug = [chip]
    for k in [1, 2, 3]:
        aug.append(np.rot90(chip, k=k, axes=(0, 1)))
    aug.append(np.flip(chip, axis=0))
    aug.append(np.flip(chip, axis=1))
    return aug


year_folders = sorted([f for f in os.listdir(data_dir)
                        if os.path.isdir(os.path.join(data_dir, f))])
print(f"Found {len(year_folders)} year folders: {year_folders}")
print(f"Training years only: {TRAINING_YEARS}\n")

all_chips = {cls: [] for cls in CLASSES.values()}

for year_folder in year_folders:
    year_code = int(year_folder[-2:])

    if year_code not in TRAINING_YEARS:
        print(f"{year_folder}: prediction-only year — skipping")
        continue

    year_dir  = os.path.join(data_dir, year_folder)
    tif_files = os.listdir(year_dir)

    # Select polygons for this year (year-specific + permanent year=00 polygons)
    year_gdf = gdf[gdf["year"].isin([year_code, 0])]
    if year_gdf.empty:
        print(f"{year_folder}: no polygons — skipping")
        continue

    # Locate all 7 band files for this year
    band_paths = []
    for suffix in BAND_SUFFIXES:
        match = next((f for f in tif_files if f.endswith(suffix)), None)
        if match is None:
            print(f"{year_folder}: missing {suffix} — skipping")
            break
        band_paths.append(os.path.join(year_dir, match))
    if len(band_paths) != 7:
        continue

    # Read spatial transform from the first band
    with rasterio.open(band_paths[0]) as src:
        transform = src.transform

    # Stack and scale all 7 bands into a single (7, H, W) array
    bands = []
    for i, bp in enumerate(band_paths):
        with rasterio.open(bp) as src:
            b = src.read(1).astype("float32")
        # Bands 0–4 (B2–B6): apply Landsat scaling; bands 5–6 (NDVI/NDWI): clip to [-1, 1]
        b = np.clip(b * SCALE + OFFSET, 0, 1) if i < 5 else np.clip(b, -1, 1)
        bands.append(b)
    stacked = np.stack(bands, axis=0)

    year_counts = {cls: 0 for cls in CLASSES.values()}
    for _, row in year_gdf.iterrows():
        cls_name = CLASSES[int(row["type"])]
        for chip in extract_chips(stacked, transform, row.geometry,
                                  CHIP_SIZE, COVERAGE_THRESHOLD):
            if cls_name == "delta":
                # Augment delta chips ×6 to help address class imbalance
                for aug in augment_chip(chip):
                    all_chips["delta"].append(aug)
                    year_counts["delta"] += 1
            else:
                all_chips["not_delta"].append(chip)
                year_counts["not_delta"] += 1

    print(f"{year_folder}: delta={year_counts['delta']}  |  not_delta={year_counts['not_delta']}")

print("")

# Cap not_delta to keep the class ratio manageable
n_delta = len(all_chips["delta"])
max_nd  = n_delta * MAX_NOT_DELTA_MULTIPLIER
if len(all_chips["not_delta"]) > max_nd:
    print(f"not_delta count {len(all_chips['not_delta'])}")
    random.shuffle(all_chips["not_delta"])
    all_chips["not_delta"] = all_chips["not_delta"][:max_nd]
    print(f"not_delta capped to {max_nd} ({MAX_NOT_DELTA_MULTIPLIER}x delta)")
print(f"Total — delta: {n_delta}  |  not_delta: {len(all_chips['not_delta'])}")

Found 14 year folders: ['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']
Training years only: [13, 16, 20, 22]

2013: delta=48  |  not_delta=554
2014: prediction-only year — skipping
2015: prediction-only year — skipping
2016: delta=36  |  not_delta=567
2017: prediction-only year — skipping
2018: prediction-only year — skipping
2019: prediction-only year — skipping
2020: delta=102  |  not_delta=533
2021: prediction-only year — skipping
2022: delta=30  |  not_delta=513
2023: prediction-only year — skipping
2024: prediction-only year — skipping
2025: prediction-only year — skipping
2026: prediction-only year — skipping

not_delta count 2167
not_delta capped to 864 (4x delta)
Total — delta: 216  |  not_delta: 864


### Save
Shuffle and split each class into train / valid / test, then write each chip to disk as a `.npy` file.

In [4]:
chip_counter = 0
for cls_name in CLASSES.values():
    chips = all_chips[cls_name]
    random.shuffle(chips)
    n       = len(chips)
    n_train = int(n * TRAIN_RATIO)
    n_valid = int(n * VALID_RATIO)
    splits  = {
        "train": chips[:n_train],
        "valid": chips[n_train:n_train + n_valid],
        "test" : chips[n_train + n_valid:]
    }
    for split, split_chips in splits.items():
        out_dir = os.path.join(chips_dir, split, cls_name)
        for chip in split_chips:
            np.save(os.path.join(out_dir, f"{cls_name}_{chip_counter:06d}.npy"), chip)
            chip_counter += 1
        print(f"  {split:5s} / {cls_name:9s} : {len(split_chips)} chips")

print("\nDone. Ready for DeltaCNN.ipynb")

  train / not_delta : 604 chips
  valid / not_delta : 129 chips
  test  / not_delta : 131 chips
  train / delta     : 151 chips
  valid / delta     : 32 chips
  test  / delta     : 33 chips

Done. Ready for DeltaCNN.ipynb
